# Etapa 4 — Explicabilidade: Feature Maps + Grad-CAM (Kaggle)

Este notebook **re-treina só o ResNet50** (o melhor modelo, ~15 min) e usa ele na hora para a explicabilidade — sem depender de checkpoint salvo antes.

Configure: **GPU On**, **Internet On**, e **Add Input** com o dataset do `pathmnist_224.npz`.

## 1. Setup (acha a pasta do repo sozinho, qualquer que seja o nome)

In [ ]:
import os, sys, glob, torch
%cd /kaggle/working
REPO_URL = 'https://github.com/SEU_USUARIO/SEU_REPO.git'  # <-- EDITE
hit = glob.glob('/kaggle/working/**/src/utils/seeds.py', recursive=True)
if not hit:
    !git clone $REPO_URL
    hit = glob.glob('/kaggle/working/**/src/utils/seeds.py', recursive=True)
ROOT = hit[0].split('/src/')[0]
os.chdir(ROOT); sys.path.insert(0, ROOT)
print('raiz do projeto:', ROOT)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
!pip install -q medmnist

## 2. DataLoaders 224×224

In [ ]:
from src.utils.seeds import set_seed
from src.etapa2_pytorch.data_224 import make_loaders
set_seed(42)
DATA_ROOT = os.path.dirname(glob.glob('/kaggle/input/**/pathmnist_224.npz', recursive=True)[0])
train_loader, val_loader, test_loader = make_loaders(size=224, batch_size=64, num_workers=2, root=DATA_ROOT)
print('ok, batches treino:', len(train_loader))

## 3. Re-treinar o ResNet50 (melhor config: SGD, lr=0.01, fine-tuning)
Salva o checkpoint num caminho ABSOLUTO em /kaggle/working/results — não se perde.

In [ ]:
from src.etapa3_cnn_vit.models import build_model
from src.etapa3_cnn_vit.run_experiments import make_optimizer
from src.etapa3_cnn_vit.engine import train_model

CKPT_DIR = '/kaggle/working/results/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
ckpt = os.path.join(CKPT_DIR, 'resnet50_fine_tuning.pt')

model = build_model('resnet50', modo='fine_tuning', pretrained=True)
opt = make_optimizer('sgd', model, 0.01)
res = train_model(model, train_loader, val_loader, optimizer=opt, epochs=6, ckpt_path=ckpt)
print('melhor val_acc:', res['best_val_acc'], '| tempo(s):', res['tempo_s'], '| VRAM(MB):', res['vram_mb'])
# garante usar os pesos do melhor epoch para a explicabilidade
model.load_state_dict(torch.load(ckpt)); model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'; model.to(device)

## 4. Feature Maps da primeira camada convolucional
Mostra o que os filtros da `conv1` capturam (bordas, texturas, cor) numa imagem de validação.

In [ ]:
from src.etapa4_xai.feature_maps import feature_maps, plot_feature_maps
os.makedirs('results/figures', exist_ok=True)
x0, y0 = val_loader.dataset[0]
fm = feature_maps(model, x0, model.conv1, max_maps=16)
plot_feature_maps(fm, 'results/figures/etapa4_feature_maps.png')

## 5. Grad-CAM: selecionar e plotar os casos exigidos
5 acertos confiantes, 5 erros grosseiros (confiantes mas errados) e 1 acerto 'por sorte'.

In [ ]:
from src.etapa4_xai.gradcam import GradCAM, gather_predictions, select_cases
from src.etapa4_xai.feature_maps import plot_gradcam_grid

linhas = gather_predictions(model, val_loader, device)
casos = select_cases(linhas, n_acertos=5, n_erros=5)

cam_obj = GradCAM(model, model.layer4[-1])   # ultima camada conv do ResNet50
plot_gradcam_grid(cam_obj, val_loader.dataset, casos['acertos_confiantes'], device,
                  'results/figures/etapa4_gradcam_acertos.png', 'Acertos confiantes')
plot_gradcam_grid(cam_obj, val_loader.dataset, casos['erros_grosseiros'], device,
                  'results/figures/etapa4_gradcam_erros.png', 'Erros grosseiros')
plot_gradcam_grid(cam_obj, val_loader.dataset, casos['acerto_por_sorte'], device,
                  'results/figures/etapa4_gradcam_sorte.png', 'Acerto por sorte')
cam_obj.remove()
print('casos:', {k: len(v) for k,v in casos.items()})

## 6. Salvar tudo
Faça **Save Version → Save & Run All** para preservar as figuras de `results/figures/`. Depois baixe-as para o relatório e o repositório.

In [ ]:
from IPython.display import Image as I, display
for f in ['etapa4_feature_maps','etapa4_gradcam_acertos','etapa4_gradcam_erros','etapa4_gradcam_sorte']:
    p = f'results/figures/{f}.png'
    print(p); display(I(p))